In [2]:
#  Imports and Setup
import sys
sys.path.append('..')  
import os, sys
sys.path.append(os.path.abspath("../src"))


In [3]:
"""
Prompt Engineering-RAG Experiments
Test different system prompts to improve RAG responses
"""

from src.rag_pipeline import RAGPipeline
from src.llm_interface import LLMInterface
import pandas as pd
from datetime import datetime
#  Initialize RAG Pipeline
print("Initializing RAG pipeline")
rag = RAGPipeline()

# Ingest a test document (make sure you have a document!)
# Replace with your actual document path
test_doc = "data/raw/sample.pdf"  # Change this to your document

try:
    result = rag.ingest_document(test_doc)
    print(f" Document ingested: {result['chunks_created']} chunks created")
except Exception as e:
    print(f" Error ingesting document: {e}")
    print("Please make sure you have a document in data/raw/")


e:\Learningskills\intellidoc-rag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Initializing RAG pipeline
RAG Pipeline initialized
Processing document: data/raw/sample.pdf
 Error ingesting document: [Errno 2] No such file or directory: 'data/raw/sample.pdf'
Please make sure you have a document in data/raw/


In [5]:
# Define Test Prompts that guide the LLM's behavior

prompt1_basic = "Answer the question based on context."

prompt2_detailed = """You are an expert assistant. 
Rules:
1. Only use provided context
2. Cite sources
3. Be concise
"""

prompt3_fewshot = """You are a helpful assistant. 

Example 1:
Q: What is RAG?
Context: RAG stands for Retrieval-Augmented Generation...
A: According to the context, RAG stands for Retrieval-Augmented Generation [Source 1]...

Now answer the user's question following this format.
"""

prompt4_strict = """You are a precise information retrieval assistant.

CRITICAL RULES:
1. ONLY use information from the provided context
2. If the answer is not in the context, say "I don't have enough information to answer that."
3. Cite sources using [Source X] notation
4. Be concise but complete
5. Do NOT make assumptions or add information not in the context
"""

prompt5_conversational = """You are a friendly, helpful assistant.

Your goal:
- Answer questions based on the provided context
- Explain clearly and simply
- Cite your sources naturally
- If information is missing, be honest about it
"""

# Store all prompts for testing
prompts = {
    "Basic": prompt1_basic,
    "Detailed": prompt2_detailed,
    "Few-shot": prompt3_fewshot,
    "Strict": prompt4_strict,
    "Conversational": prompt5_conversational
}

print(f" Defined {len(prompts)} prompt variants")

 Defined 5 prompt variants


In [6]:
#  Define Test Questions

test_questions = [
    "What is the main topic of this document?",
    "Who are the authors?",
    "What are the key findings?",
    "What methodology was used?",
]

print(f" Defined {len(test_questions)} test questions")


 Defined 4 test questions


In [7]:
# Run Experiments

print("STARTING PROMPT EXPERIMENTS")
results = []

for question in test_questions:
    print(f"\n Question: {question}")
     
    for prompt_name, system_prompt in prompts.items():
        print(f"\n Testing prompt: {prompt_name}")
        
        try:
            # Temporarily change the system prompt
            original_llm = rag.llm
            rag.llm = LLMInterface(model="llama3", temperature=0.4)
            
            # Get query embedding
            query_embedding = rag.embedding_gen.generate_embedding(question)
            
            # Retrieve relevant documents
            search_results = rag.vector_store.search(query_embedding, n_results=3)
            
            # Prepare context
            context = []
            for i, doc in enumerate(search_results['documents'][0]):
                context.append({
                    'content': doc,
                    'metadata': search_results['metadatas'][0][i]
                })
            
            # Generate answer with custom system prompt
            answer = rag.llm.generate_response(
                question, 
                [ctx['content'] for ctx in context],
                system_prompt=system_prompt
            )
            
            # Store result
            results.append({
                'question': question,
                'prompt_type': prompt_name,
                'answer': answer,
                'answer_length': len(answer),
                'timestamp': datetime.now()
            })
            
            # Display answer 
            print(f"Answer: {answer[:200]}..." if len(answer) > 200 else f"Answer: {answer}")
            
        except Exception as e:
            print(f" Error: {e}")
            results.append({
                'question': question,
                'prompt_type': prompt_name,
                'answer': f"ERROR: {str(e)}",
                'answer_length': 0,
                'timestamp': datetime.now()
            })


print(" EXPERIMENTS COMPLETE")



STARTING PROMPT EXPERIMENTS

 Question: What is the main topic of this document?

 Testing prompt: Basic
Answer: Based on the provided context, there is no document provided, so it's not possible to determine the main topic. If you provide the actual text or content of a document, I'd be happy to help answer you...

 Testing prompt: Detailed
Answer: Since there is no provided text or document, it's not possible to determine the main topic. Would you like to provide more context or information about the document you're referring to? I'd be happy t...

 Testing prompt: Few-shot
Answer: According to the context, there is no explicit mention of a specific topic. The context appears to be blank or empty. Therefore, it is not possible to determine the main topic of this document based o...

 Testing prompt: Strict
Answer: Based on the provided context, I don't have enough information to answer that. Can you please provide more context or information about the document?

 Testing prompt: Conv

KeyboardInterrupt: 

In [8]:
#  Analyze Results
print("\n ANALYSIS")

df = pd.DataFrame(results)
df
# Group by prompt type
print("\n1. Average Answer Length by Prompt Type:")
print(df.groupby('prompt_type')['answer_length'].mean().sort_values(ascending=False))

print("\n2. Results Summary:")
print(df[['question', 'prompt_type', 'answer_length']])


 ANALYSIS

1. Average Answer Length by Prompt Type:
prompt_type
Few-shot          266.500000
Basic             237.333333
Conversational    217.500000
Detailed          185.000000
Strict            133.000000
Name: answer_length, dtype: float64

2. Results Summary:
                                    question     prompt_type  answer_length
0   What is the main topic of this document?           Basic            211
1   What is the main topic of this document?        Detailed            207
2   What is the main topic of this document?        Few-shot            323
3   What is the main topic of this document?          Strict            149
4   What is the main topic of this document?  Conversational            141
5                       Who are the authors?           Basic            261
6                       Who are the authors?        Detailed            163
7                       Who are the authors?        Few-shot            210
8                       Who are the authors?     

In [9]:
#  Side-by-Side Comparison


comparison_question = test_questions[0]  # First question
print(f"\n DETAILED COMPARISON")
print(f"Question: {comparison_question}")

comparison_df = df[df['question'] == comparison_question]

for idx, row in comparison_df.iterrows():
    
    print(f"Prompt: {row['prompt_type']}")
    print(f"Answer:\n{row['answer']}\n")
    print(f"Length: {row['answer_length']} characters")

# Cell 8: Evaluation Criteria
print("\n EVALUATION CRITERIA")


print("""
Manually evaluate each answer on:

1.  Accuracy: Does it answer the question correctly?
2.  Relevance: Is the answer focused on the question?
3.  Source Citation: Does it cite sources properly?
4.  Hallucination: Does it make up information?
5.  Length: Is it concise but complete?
6.  Tone: Is the tone appropriate?

Rate each prompt on a scale of 1-5 for each criteria.
""")

# Create evaluation template
eval_template = pd.DataFrame({
    'Prompt': list(prompts.keys()),
    'Accuracy (1-5)': [0] * len(prompts),
    'Relevance (1-5)': [0] * len(prompts),
    'Citations (1-5)': [0] * len(prompts),
    'No Hallucination (1-5)': [0] * len(prompts),
    'Conciseness (1-5)': [0] * len(prompts),
    'Tone (1-5)': [0] * len(prompts),
})

print("\nEvaluation Template:")
print(eval_template)

# Save Results

output_file = f"prompt_experiment_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
df.to_csv(output_file, index=False)
print(f"\n Results saved to: {output_file}")

# Recommendations
print("\n NEXT STEPS")
print("""
Based on your experiments:

1. Identify which prompt performed best
2. Test that prompt with more questions
3. Iterate on the winning prompt
4. Add your best prompt to src/llm_interface.py as default

Typical findings:
- "Basic" prompts often lack structure
- "Detailed" prompts improve citation
- "Strict" prompts reduce hallucination
- "Few-shot" helps with formatting
- "Conversational" improves readability


""")


 DETAILED COMPARISON
Question: What is the main topic of this document?
Prompt: Basic
Answer:
Based on the provided context, there is no document provided, so it's not possible to determine the main topic. If you provide the actual text or content of a document, I'd be happy to help answer your question!

Length: 211 characters
Prompt: Detailed
Answer:
Since there is no provided text or document, it's not possible to determine the main topic. Would you like to provide more context or information about the document you're referring to? I'd be happy to help!

Length: 207 characters
Prompt: Few-shot
Answer:
According to the context, there is no explicit mention of a specific topic. The context appears to be blank or empty. Therefore, it is not possible to determine the main topic of this document based on the provided information. If you could provide more context or details, I'd be happy to help you identify the main topic!

Length: 323 characters
Prompt: Strict
Answer:
Based on the pro